In [2]:
# sql저장용 파일 정제 코드

import pandas as pd
import csv

src = r"C:\py_temp\3proj\fire_rescue_event_20241231.csv"
dst = r"C:\py_temp\3proj\fire_rescue_event_mysql_clean.csv"

df = None

# 인코딩 순서대로 시도
for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
    try:
        df = pd.read_csv(src, encoding=enc, dtype=str, engine="python")
        print("읽기 성공 인코딩:", enc)
        break
    except Exception as e:
        print("실패 인코딩:", enc, "|", e)

if df is None:
    raise Exception("파일 읽기 실패")

df = df.fillna("")

print("원본 행수:", len(df))
print("원본 컬럼명:", list(df.columns))
print(df.head())

# 모든 문자 컬럼 정리
for c in df.columns:
    df[c] = (
        df[c].astype(str)
        .str.replace("\r", " ", regex=False)
        .str.replace("\n", " ", regex=False)
        .str.replace("\t", " ", regex=False)
        .str.strip()
    )

# MySQL 적재용으로 깨끗하게 재저장
df.to_csv(
    dst,
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n"
)

print("저장 완료:", dst)
print("저장 행수:", len(df))

읽기 성공 인코딩: utf-8-sig
원본 행수: 885541
원본 컬럼명: ['번호', '신고년월일', '신고시각', '출동년월일', '출동시각', '발생장소_시', '발생장소_구', '발생장소_동', '사고원인', '사고원인코드명_사고종별']
  번호       신고년월일  신고시각       출동년월일  출동시각   발생장소_시 발생장소_구 발생장소_동  사고원인  \
0  1  2024-01-01  0:01  2024-01-01  0:02  강원특별자치도    원주시    봉산동    화재   
1  2  2024-01-01  0:04  2024-01-01  0:04      경기도    김포시    구래동    화재   
2  3  2024-01-01  0:04  2024-01-01  0:04     경상북도    구미시    구평동  자살추정   
3  4  2024-01-01  0:04  2024-01-01  0:04     경상북도    구미시    구평동  자살추정   
4  5  2024-01-01  0:04  2024-01-01  0:05      경기도    연천군    전곡읍    교통   

         사고원인코드명_사고종별  
0                화재확인  
1                화재확인  
2         연탄·번개탄·가스중독  
3         연탄·번개탄·가스중독  
4  차량(버스, 화물, 승용, 승합)  
저장 완료: C:\py_temp\3proj\fire_rescue_event_mysql_clean.csv
저장 행수: 885541


In [45]:
print(admin_df[["행정동코드", "법정동코드"]].isna().sum())
print(event_df[["행정동코드", "법정동코드"]].isna().sum())
print(cctv_df[["행정동코드", "법정동코드"]].isna().sum())

행정동코드    129
법정동코드    129
dtype: int64
행정동코드    20677
법정동코드    20677
dtype: int64
행정동코드    11231
법정동코드    11231
dtype: int64


In [46]:
print(cctv_df[cctv_df["행정동코드"].isna()][["소재지지번주소", "소재지도로명주소"]].head(50))

                         소재지지번주소                              소재지도로명주소
125700  경기도 수원시 팔달구 매산로 3가 122-5                    경기도 수원시 팔달구 향교로 93
125881     경기도 수원시 권선구 권선로596번길8                 경기도 수원시 권선구 세류동 837-1
126937           경기도 화성시 황계동 209                       경기도 화성시 황계동 209
126961    경기도 수언시 권선구 세류동 344-13               경기도 수원시 권선구 권선로544번길 20
127103     경기도 수원시 영통구 우만동 산55-5                 경기도 수원시 영통구 우만동 산55-5
127134      경기도 수원시 장안구 파장천로 143                  경기도 수원시 장안구 파장천로 143
128216  경기도 수원시 팔달구 매산로 3가 122-5                    경기도 수원시 팔달구 향교로 93
128515      경기도 수원시 장안구 파장천로 143                  경기도 수원시 장안구 파장천로 143
128636        경기도 수원시 정자2동 48-13                     경기도 수원시 장안구 장안로98
128957     경기도 수원시 장안구 율쳔동 356-6               경기도 수원시 장안구 화산로285번길 27
129065          경기도 수원시 호매실동 6-9                      경기도 수원시 호매실동 6-9
129302   경기도 수원시 영통구 메탄동 153-190               경기도 수원시 영통구 메탄동 153-190
130263         경기도 성남시 궁내동 344-1                         경기도 성남시 궁내로 8
130265

In [51]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import re

# =========================
# 0. 사용자 설정
# =========================
MYSQL_USER = "root"
MYSQL_PASSWORD = input('비밀번호: ')
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DB = "cctv_project"

BASE_DIR = r"C:\py_temp\3proj"
ADMIN_PATH = BASE_DIR + r"\행정구역현황_경기도.csv"
KIKMIX_PATH = BASE_DIR + r"\KIKmix.20260201.xlsx"

OUT_ADMIN_CODED = BASE_DIR + r"\admin_with_2codes.csv"
OUT_EVENT_CODED = BASE_DIR + r"\fire_event_with_2codes.csv"
OUT_CCTV_CODED = BASE_DIR + r"\cctv_with_2codes.csv"
OUT_FINAL_ADMINKEY = BASE_DIR + r"\final_join_by_admin_code.csv"

CCTV_SCHEMA = "cctv_project"
CCTV_TABLE = "cctv_raw"
EVENT_SCHEMA = "fire_rescue"
EVENT_TABLE = "fire_event_raw"

engine = create_engine(
    f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}?charset=utf8mb4"
)

# =========================
# 1. KIKmix 로드
# =========================
kik_df = pd.read_excel(KIKMIX_PATH)

kik_df = kik_df.rename(columns={
    "시도명": "시도명",
    "시군구명": "시군구명",
    "읍면동명": "행정동명",
    "행정동코드": "행정동코드",
    "동리명": "법정동명",
    "법정동코드": "법정동코드",
    "말소일자": "말소일자"
}).copy()

for col in kik_df.columns:
    if kik_df[col].dtype == "object":
        kik_df[col] = kik_df[col].astype(str).str.strip()
        kik_df[col] = kik_df[col].str.replace(r"\s+", " ", regex=True)

kik_df = kik_df[kik_df["시도명"].eq("경기도")].copy()

kik_df["행정동코드"] = kik_df["행정동코드"].astype(str).str.replace(".0", "", regex=False).str.strip()
kik_df["법정동코드"] = kik_df["법정동코드"].astype(str).str.replace(".0", "", regex=False).str.strip()

if "말소일자" in kik_df.columns:
    kik_df["말소일자"] = kik_df["말소일자"].replace(["", "nan", "None"], np.nan)
    kik_live = kik_df[kik_df["말소일자"].isna()].copy()
else:
    kik_live = kik_df.copy()

# 중복 이름은 첫 번째 코드 사용
map_bjdname_to_bjdcode = (
    kik_live.drop_duplicates(subset=["법정동명"], keep="first")
    .set_index("법정동명")["법정동코드"]
    .to_dict()
)

map_bjdname_to_admincode = (
    kik_live.drop_duplicates(subset=["법정동명"], keep="first")
    .set_index("법정동명")["행정동코드"]
    .to_dict()
)

map_adminname_to_admincode = (
    kik_live.drop_duplicates(subset=["행정동명"], keep="first")
    .set_index("행정동명")["행정동코드"]
    .to_dict()
)

map_adminname_to_bjdcode = (
    kik_live.drop_duplicates(subset=["행정동명"], keep="first")
    .set_index("행정동명")["법정동코드"]
    .to_dict()
)

# 읍면동 후보명 목록
dong_name_set = set(kik_live["법정동명"].dropna().astype(str).tolist()) | set(kik_live["행정동명"].dropna().astype(str).tolist())
dong_name_list = sorted([x for x in dong_name_set if x not in ["", "nan", "None"]], key=len, reverse=True)
dong_pattern = "|".join(re.escape(x) for x in dong_name_list)

# 오타 보정
replace_map = {
    "수언시": "수원시",
    "율쳔동": "율전동",
    "메탄동": "매탄동",
}

# =========================
# 2. 인구/면적 테이블
# 읍면동명만으로 매칭
# =========================
try:
    admin_df = pd.read_csv(ADMIN_PATH, encoding="utf-8")
except:
    admin_df = pd.read_csv(ADMIN_PATH, encoding="cp949")

admin_df = admin_df.rename(columns={
    "기준년도": "기준년도",
    "시도명": "시도명",
    "시군구명": "시군구명",
    "읍면동명": "읍면동명",
    "면적(k㎡)": "면적_km2",
    "면적(㎢)": "면적_km2",
    "세대수": "세대수",
    "인구수": "인구수"
}).copy()

for col in admin_df.columns:
    if admin_df[col].dtype == "object":
        admin_df[col] = admin_df[col].astype(str).str.strip()
        admin_df[col] = admin_df[col].replace(replace_map, regex=True)

admin_df = admin_df[admin_df["시도명"].eq("경기도")].copy()

dong_only = admin_df["읍면동명"].fillna("").astype(str).str.strip()

# 법정동명 우선
admin_df["법정동코드"] = dong_only.map(map_bjdname_to_bjdcode)
admin_df["행정동코드"] = dong_only.map(map_bjdname_to_admincode)

# 안 되면 행정동명
mask = admin_df["행정동코드"].isna()
admin_df.loc[mask, "행정동코드"] = dong_only[mask].map(map_adminname_to_admincode)
admin_df.loc[mask, "법정동코드"] = dong_only[mask].map(map_adminname_to_bjdcode)

admin_df.to_csv(OUT_ADMIN_CODED, index=False, encoding="utf-8-sig")
print("\n[admin 결측]")
print(admin_df[["행정동코드", "법정동코드"]].isna().sum())

# =========================
# 3. 사건사고 테이블
# 발생장소_동만으로 매칭
# =========================
sql_event = f"""
SELECT
    번호,
    신고년월일,
    신고시각,
    출동년월일,
    출동시각,
    발생장소_시,
    발생장소_구,
    발생장소_동,
    사고원인,
    사고원인코드명_사고종별
FROM {EVENT_SCHEMA}.{EVENT_TABLE}
"""

event_df = pd.read_sql(sql_event, engine)

for col in event_df.columns:
    if event_df[col].dtype == "object":
        event_df[col] = event_df[col].astype(str).str.strip()
        event_df[col] = event_df[col].replace(replace_map, regex=True)

event_df = event_df[event_df["발생장소_시"].eq("경기도")].copy()

dong_only = event_df["발생장소_동"].fillna("").astype(str).str.strip()

# 법정동명 우선
event_df["법정동코드"] = dong_only.map(map_bjdname_to_bjdcode)
event_df["행정동코드"] = dong_only.map(map_bjdname_to_admincode)

# 안 되면 행정동명
mask = event_df["행정동코드"].isna()
event_df.loc[mask, "행정동코드"] = dong_only[mask].map(map_adminname_to_admincode)
event_df.loc[mask, "법정동코드"] = dong_only[mask].map(map_adminname_to_bjdcode)

event_df.to_csv(OUT_EVENT_CODED, index=False, encoding="utf-8-sig")
print("\n[event 결측]")
print(event_df[["행정동코드", "법정동코드"]].isna().sum())

# =========================
# 4. CCTV 테이블
# 주소에서 읍면동만 추출 후 매칭
# 지번주소 우선 > 도로명주소 괄호 > 도로명주소 전체
# =========================
sql_cctv = f"""
SELECT
    관리번호,
    관리기관명,
    소재지도로명주소,
    소재지지번주소,
    카메라대수,
    설치목적구분
FROM {CCTV_SCHEMA}.{CCTV_TABLE}
"""

cctv_df = pd.read_sql(sql_cctv, engine)

for col in cctv_df.columns:
    if cctv_df[col].dtype == "object":
        cctv_df[col] = cctv_df[col].astype(str).str.strip()
        cctv_df[col] = cctv_df[col].replace(replace_map, regex=True)

jibun = cctv_df["소재지지번주소"].fillna("").astype(str).str.strip()
road = cctv_df["소재지도로명주소"].fillna("").astype(str).str.strip()

# 진짜 경기도만
mask_gyeonggi = jibun.str.startswith("경기도 ") | road.str.startswith("경기도 ")
cctv_df = cctv_df[mask_gyeonggi].copy()
jibun = jibun[mask_gyeonggi]
road = road[mask_gyeonggi]

# 지번주소에서 읍면동
dong_jibun = jibun.str.extract(f"({dong_pattern})", expand=False)

# 도로명주소 괄호 안 읍면동
dong_paren = road.str.extract(r"\(([가-힣0-9]+[동읍면리])", expand=False)

# 도로명주소 전체에서 읍면동
dong_road = road.str.extract(f"({dong_pattern})", expand=False)

# 우선순위
dong_only = dong_jibun.where(dong_jibun.notna(), dong_paren)
dong_only = dong_only.where(dong_only.notna(), dong_road)

# 법정동명 우선
cctv_df["법정동코드"] = dong_only.map(map_bjdname_to_bjdcode)
cctv_df["행정동코드"] = dong_only.map(map_bjdname_to_admincode)

# 안 되면 행정동명
mask = cctv_df["행정동코드"].isna()
cctv_df.loc[mask, "행정동코드"] = dong_only[mask].map(map_adminname_to_admincode)
cctv_df.loc[mask, "법정동코드"] = dong_only[mask].map(map_adminname_to_bjdcode)

cctv_df.to_csv(OUT_CCTV_CODED, index=False, encoding="utf-8-sig")
print("\n[cctv 결측]")
print(cctv_df[["행정동코드", "법정동코드"]].isna().sum())

print("\n[cctv 미매칭 샘플]")
print(cctv_df[cctv_df["행정동코드"].isna()][["소재지지번주소", "소재지도로명주소"]].head(50))

비밀번호:  kmsoo1995!!



[admin 결측]
행정동코드    29
법정동코드    29
dtype: int64

[event 결측]
행정동코드    3959
법정동코드    3959
dtype: int64

[cctv 결측]
행정동코드    285
법정동코드    285
dtype: int64

[cctv 미매칭 샘플]
       소재지지번주소                                           소재지도로명주소
144449                              경기도 안산시 단원구 고잔동 681-4 (중앙분리대)
144534                         경기도 안산시 단원구 고잔동 766-2(안산성마리아성당맞은편)
145517                              경기도 안산시 단원구 와동 707-16(열녀문사거리)
146039                      경기도 안산시 단원구 고잔동 729-5 아도르웨딩홀 앞(중앙분리대)
147215                        경기도 고양시 일산동구 정발산동 1201-4 (저동중 후문 앞)
147221                         경기도 고양시 일산동구 정발산동 1166-7 (케논대리점 앞)
147259                경기도 고양시 덕양구 덕은동 120-10 (제일카센터삼거리, 대덕동자율방범대)
147264                              경기도 고양시 덕양구 행신동  1017 (행신사거리)
147298                     경기도 고양시 일산동구 정발산동 1159 두루미공원 일대(저동중인근)
147304                           경기도 고양시 일산동구 장항동 750-1 (대우메종리브르)
147305                   경기도 고양시 일산동구 장항동 751 (호수삼거리, 강선17단지 삼거리)
147318                       경기도 고양시 일산동구

In [8]:
import pandas as pd
import numpy as np

In [9]:
ss = pd.read_csv('C:\\py_temp\\3proj\\fire_event_with_2codes.csv', encoding = 'utf-8-sig')

In [41]:
ss.loc[ss['발생장소_동']=='구래동',:].reset_index(drop = True)
display(ss['사고원인'].unique())

# sss = []
# for i in ss['사고원인'].unique():
#     sss.append(i)

ssss = ss.loc[(ss['사고원인'] == '자살추정') & (ss['발생장소_구'] == '성남시 분당구'),:].reset_index(drop = True)

display(ssss['사고원인코드명_사고종별'].unique())

ssss.loc[ssss['사고원인코드명_사고종별'] == '방화·분신',:].reset_index(drop = True)

ssss.loc[ssss['출동년월일'].str.contains('06-'), :]

array(['화재', '교통', '비화재보 확인', '위치확인', '잠금장치개방', '승강기', '자살추정',
       '생활끼임 안전조치', '동물처리', '테러(의심)', '인명 갇힘', '행사지원', '산악',
       '장애물 제거 및 안전조치', '피해복구 지원', '추락', '끼임', '누출사고', '붕괴·도괴(깔림)', '수난',
       '벌(집)제거', '폭발', '기타 사고', '기타 생활안전활동', '감염병 지원', '항공기 사고'],
      dtype=object)

array(['투신', '자해', '방화·분신', '기타 자살추정', '교사(목멤)', '연탄·번개탄·가스중독', '약물 과다복용',
       '음독'], dtype=object)

,번호,신고년월일,신고시각,출동년월일,출동시각,발생장소_시,발생장소_구,발생장소_동,사고원인,사고원인코드명_사고종별,법정동코드,행정동코드
79,224277,2024-06-01,11:57,2024-06-01,11:58,경기도,성남시 분당구,야탑동,자살추정,투신,4.113511e+09,4.113561e+09
80,225353,2024-06-01,21:24,2024-06-01,21:25,경기도,성남시 분당구,서현동,자살추정,투신,4.113510e+09,4.113558e+09
81,236022,2024-06-06,21:25,2024-06-06,21:25,경기도,성남시 분당구,금곡동,자살추정,자해,4.111313e+09,4.111366e+09
82,252468,2024-06-14,7:07,2024-06-14,7:08,경기도,성남시 분당구,정자동,자살추정,자해,4.111113e+09,4.111157e+09
83,263511,2024-06-18,14:14,2024-06-18,14:15,경기도,성남시 분당구,서현동,자살추정,자해,4.113510e+09,4.113558e+09
84,265040,2024-06-19,7:13,2024-06-19,7:15,경기도,성남시 분당구,야탑동,자살추정,투신,4.113511e+09,4.113561e+09
85,278721,2024-06-24,9:51,2024-06-24,9:51,경기도,성남시 분당구,운중동,자살추정,투신,4.113512e+09,4.113568e+09
86,279860,2024-06-24,15:21,2024-06-24,15:26,경기도,성남시 분당구,수내동,자살추정,자해,4.113510e+09,4.113552e+09
87,286290,2024-06-26,19:53,2024-06-26,19:53,경기도,성남시 분당구,금곡동,자살추정,투신,4.111313e+09,4.111366e+09
